# 05. SWE agent: от задачи до проверенного изменения

## Что агент пока не умеет

После `04` агент умеет делегировать анализ, но ещё не замыкает цикл изменения программного проекта.

> Coding agent полезен не тогда, когда умеет написать patch, а когда умеет понять задачу, локализовать изменение, применить его и проверить результат.


![SWE cycle](../visuals/openclaw_path/10_swe_cycle.svg)


## Preflight для повторяемой демонстрации

Перед главой:

```bash
uv run python scripts/reset_swe_demo.py
OPENCLAW_ENABLE_LOCAL_SHELL=1 uv run python scripts/preflight_openclaw_workshop.py
```

Stage 05 требует shell, потому что кульминация — запуск `pytest`.


## Acceptance criteria для SWE-задачи

```text
VK connector должен разбивать сообщения длиннее 3500 символов.

1. Каждый chunk не длиннее 3500 символов.
2. Порядок и полный текст сохраняются.
3. Пустые chunks не отправляются.
4. Каждый chunk отправляется отдельным вызовом VK API.
5. Каждый chunk получает отдельный random_id.
6. Добавить unit tests без реального сетевого запроса.
7. Запустить только релевантные pytest.
8. Перед финальным ответом передать patch-reviewer git diff и результаты тестов.
```

Сейчас дефект воспроизводим: bridge содержит `reply[:3500]`, а connector не имеет chunking helper.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, print_stage_context, register_graphs, write_text

print_stage_context(require_shell=True)


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\nfrom connectors.vk import VK_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n\n\ndef _backend(*, require_shell: bool = False):\n    root = _workspace_root()\n    shell_enabled = os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\"\n    if require_shell and not shell_enabled:\n        raise RuntimeError(\"OPENCLAW_ENABLE_LOCAL_SHELL=1 is required for this stage\")\n    if shell_enabled:\n        from deepagents.backends import LocalShellBackend\n\n        return LocalShellBackend(\n            root_dir=root,\n            virtual_mode=True,\n            inherit_env=False,\n            timeout=120,\n            max_output_bytes=80_000,\n        )\n\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nTOOLS = [*JENKINS_TOOLS, *VK_TOOLS]\nSUBAGENTS = [\n    {\"name\": \"repo-researcher\", \"description\": \"Research repository facts before implementation.\", \"system_prompt\": \"Find relevant files, APIs, tests, and risks. Cite paths.\"},\n    {\"name\": \"patch-reviewer\", \"description\": \"Review concrete diffs and test coverage.\", \"system_prompt\": \"Review correctness, regressions, tests, and security. Expect the main agent to pass git diff and test results.\"},\n]\nSWE_PROMPT = \"\"\"\\\nYou are OpenClaw SWE agent. Respond in the user's language; default to Russian.\nFollow this issue-resolution loop:\n1. Reproduce or characterize the issue.\n2. Localize relevant files and tests.\n3. Patch the root cause.\n4. Add or update regression coverage.\n5. Run narrow tests first, then related checks.\n6. Before delegating to patch-reviewer, run git diff for changed files and include the diff and test results in the reviewer task.\n\"\"\"\nswe_agent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=TOOLS,\n    system_prompt=SWE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(require_shell=True),\n)\n"
print(write_text("agents/openclaw_05_swe_agent.py", ENTRYPOINT).relative_to(REPO_ROOT))
print(register_graphs({
    "openclaw_05_swe": "./agents/openclaw_05_swe_agent.py:swe_agent",
}).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
VK connector должен разбивать сообщения длиннее 3500 символов на несколько частей. Реализуй acceptance criteria из notebook и добавь unit tests без реального VK API.
```

### Ожидаемое поведение

1. Агент локализует `connectors/vk.py`, `scripts/vk_langgraph_bridge.py`, tests.
2. Создаёт/обновляет тест на длинное сообщение.
3. Вносит минимальный patch.
4. Запускает narrow pytest.
5. Передаёт `git diff` и test results в `patch-reviewer`.

### Что изменилось относительно предыдущего этапа

Появился полный SWE workflow: issue → research → edit → test → review → summary.

### Текущее ограничение

Это учебный SWE контур. Нет production isolation, scheduler, полноценной memory subsystem и marketplace skills.
